In [29]:
import pandas as pd
import numpy as np

In [31]:
# Import Movie data -> Summary plot 

# Load parquet file into a DataFrame
df_plot = pd.read_parquet("0000.parquet")

# Keeping only American movies after 1969
df_plot_american = df_plot[(df_plot['Origin/Ethnicity']=='American') & (df_plot['Release Year']>1969)]

df_plot_american.head()


,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot,PlotSummary
8513,1970,Adam at Six A.M.,American,Robert Scheerer,"Michael Douglas, Lee Purcell, Joe Don Baker, L...",unknown,https://en.wikipedia.org/wiki/Adam_at_Six_A.M.,"The film revolves around Adam Gaines, a semant...","The film revolves around Adam Gaines, a semant..."
8514,1970,The Adventurers,American,Lewis Gilbert,"Bekim Fehmiu, Candice Bergen, Charles Aznavour...",unknown,https://en.wikipedia.org/wiki/The_Adventurers_...,Set in the fictional Latin American country of...,Set in the fictional Latin American country of...
8515,1970,Airport,American,George Seaton,"Burt Lancaster, Dean Martin, Jean Seberg, Jacq...",unknown,https://en.wikipedia.org/wiki/Airport_(1970_film),Chicago is paralyzed by a snowstorm affecting ...,Chicago is paralyzed by a snowstorm affecting ...
8516,1970,Alex in Wonderland,American,Paul Mazursky,"Donald Sutherland, Ellen Burstyn, Federico Fel...",unknown,https://en.wikipedia.org/wiki/Alex_in_Wonderland,Young director Alex Morrison feels compelled t...,Alex Morrison feels compelled to follow his re...
8517,1970,Angel Unchained,American,Lee Madden,"Larry Bishop, Tyne Daly, Aldo Ray",unknown,https://en.wikipedia.org/wiki/Angel_Unchained,"Following a gang fight, biker Angel, calls it ...","Following a gang fight, biker Angel, calls it ..."


In [32]:
# Checking Missing values
# Missing values
(df_plot_american.isna() | (df_plot_american == "Unknown")| (df_plot_american == "unknown")|(df_plot_american == "None")).sum()

Release Year          0
Title                 1
Origin/Ethnicity      0
Director            126
Cast                163
Genre               307
Wiki Page             0
Plot                  0
PlotSummary           0
dtype: int64

### One missing value for Title can be ignored, as it corresponds to the movie Unknown.

### The goal in the subsequent steps is to fill the missing values for Director, Cast (Actors), and Genres using the IMDb and movie plot data.

In [33]:
####Importing Genre, Cast, and Director from IMDB dataset###


#Importing Genre from IMDB dataset

df_Imdb_genre = pd.read_csv("title.basics.tsv.gz",sep="\t")

#Importing  Director from IMDB dataset

df_Imdb_director = pd.read_csv("title.crew.tsv.gz",sep="\t")

#Adding cast from imdb table

df_cast = pd.read_csv("title.principals.tsv.gz",sep="\t")

# Creating director and cast names data frame 

df_director_name = pd.read_csv("name.basics.tsv.gz",sep="\t")


In [34]:
### keeping only titleType = movie
#### removing rows where genres = \N
### deduping on startYear and PrimaryTitle

# only keeping titleType = movie & removing rows where genres = \N

df_Imdb_genre = df_Imdb_genre[(df_Imdb_genre.titleType=='movie') & (df_Imdb_genre.genres!="\\N")]

# remove duplicate startYear and PrimaryTitle. Keep the first one only. 
df_Imdb_genre = df_Imdb_genre.drop_duplicates(subset= ['startYear','primaryTitle'], keep='first')

In [35]:
### Format conversion 

## converting startYear into numeric

df_Imdb_genre['startYear'] = pd.to_numeric(df_Imdb_genre['startYear'], errors='coerce')

In [36]:
### Merging movie's plot with IMDB genre data

# Merging df_plot_american with df_Imdb_genere to get genere name
df_merged_genre = df_plot_american.merge(df_Imdb_genre[["primaryTitle","startYear", "genres","tconst"]], left_on=["Title",'Release Year'],right_on=['primaryTitle',"startYear"], how="left")

df_merged_genre.head(5)


,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot,PlotSummary,primaryTitle,startYear,genres,tconst
0,1970,Adam at Six A.M.,American,Robert Scheerer,"Michael Douglas, Lee Purcell, Joe Don Baker, L...",unknown,https://en.wikipedia.org/wiki/Adam_at_Six_A.M.,"The film revolves around Adam Gaines, a semant...","The film revolves around Adam Gaines, a semant...",Adam at Six A.M.,1970.0,Drama,tt0065371
1,1970,The Adventurers,American,Lewis Gilbert,"Bekim Fehmiu, Candice Bergen, Charles Aznavour...",unknown,https://en.wikipedia.org/wiki/The_Adventurers_...,Set in the fictional Latin American country of...,Set in the fictional Latin American country of...,The Adventurers,1970.0,"Action,Adventure,Drama",tt0065374
2,1970,Airport,American,George Seaton,"Burt Lancaster, Dean Martin, Jean Seberg, Jacq...",unknown,https://en.wikipedia.org/wiki/Airport_(1970_film),Chicago is paralyzed by a snowstorm affecting ...,Chicago is paralyzed by a snowstorm affecting ...,Airport,1970.0,"Action,Drama,Thriller",tt0065377
3,1970,Alex in Wonderland,American,Paul Mazursky,"Donald Sutherland, Ellen Burstyn, Federico Fel...",unknown,https://en.wikipedia.org/wiki/Alex_in_Wonderland,Young director Alex Morrison feels compelled t...,Alex Morrison feels compelled to follow his re...,Alex in Wonderland,1970.0,"Comedy,Drama",tt0065380
4,1970,Angel Unchained,American,Lee Madden,"Larry Bishop, Tyne Daly, Aldo Ray",unknown,https://en.wikipedia.org/wiki/Angel_Unchained,"Following a gang fight, biker Angel, calls it ...","Following a gang fight, biker Angel, calls it ...",Angel Unchained,1970.0,"Action,Drama,Thriller",tt0065401


In [37]:
### Fallback to plot-dataset genre values 

# Combining genre and genres such that first generes is picked if it's not null, else genre

df_merged_genre['final_genre'] = df_merged_genre['genres'].combine_first(df_merged_genre['Genre'])

df_merged_genre.head(5)

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot,PlotSummary,primaryTitle,startYear,genres,tconst,final_genre
0,1970,Adam at Six A.M.,American,Robert Scheerer,"Michael Douglas, Lee Purcell, Joe Don Baker, L...",unknown,https://en.wikipedia.org/wiki/Adam_at_Six_A.M.,"The film revolves around Adam Gaines, a semant...","The film revolves around Adam Gaines, a semant...",Adam at Six A.M.,1970.0,Drama,tt0065371,Drama
1,1970,The Adventurers,American,Lewis Gilbert,"Bekim Fehmiu, Candice Bergen, Charles Aznavour...",unknown,https://en.wikipedia.org/wiki/The_Adventurers_...,Set in the fictional Latin American country of...,Set in the fictional Latin American country of...,The Adventurers,1970.0,"Action,Adventure,Drama",tt0065374,"Action,Adventure,Drama"
2,1970,Airport,American,George Seaton,"Burt Lancaster, Dean Martin, Jean Seberg, Jacq...",unknown,https://en.wikipedia.org/wiki/Airport_(1970_film),Chicago is paralyzed by a snowstorm affecting ...,Chicago is paralyzed by a snowstorm affecting ...,Airport,1970.0,"Action,Drama,Thriller",tt0065377,"Action,Drama,Thriller"
3,1970,Alex in Wonderland,American,Paul Mazursky,"Donald Sutherland, Ellen Burstyn, Federico Fel...",unknown,https://en.wikipedia.org/wiki/Alex_in_Wonderland,Young director Alex Morrison feels compelled t...,Alex Morrison feels compelled to follow his re...,Alex in Wonderland,1970.0,"Comedy,Drama",tt0065380,"Comedy,Drama"
4,1970,Angel Unchained,American,Lee Madden,"Larry Bishop, Tyne Daly, Aldo Ray",unknown,https://en.wikipedia.org/wiki/Angel_Unchained,"Following a gang fight, biker Angel, calls it ...","Following a gang fight, biker Angel, calls it ...",Angel Unchained,1970.0,"Action,Drama,Thriller",tt0065401,"Action,Drama,Thriller"


In [38]:
### Checking missing values in the df_merged

(df_merged_genre.isna() | (df_merged_genre == "Unknown")| (df_merged_genre == "unknown")|(df_merged_genre == "None")).sum()

# only 55 unknown genre are remaining ( from the initial 307 )

Release Year           0
Title                  1
Origin/Ethnicity       0
Director             126
Cast                 163
Genre                307
Wiki Page              0
Plot                   0
PlotSummary            0
primaryTitle        1611
startYear           1610
genres              1610
tconst              1610
final_genre           55
dtype: int64

In [39]:
### Merging movie's plot with IMDB Directos data
df_merged_genre = df_merged_genre.merge(df_Imdb_director[['tconst','directors']], left_on =['tconst'], right_on=['tconst'],how='left')
df_merged_genre.shape

(8864, 15)

In [40]:
### Adding director name to the merged dataset

df_merged_genre = df_merged_genre.merge(df_director_name[['nconst','primaryName']], left_on =['directors'],right_on=['nconst'],how='left')
df_merged_genre.head(5)

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot,PlotSummary,primaryTitle,startYear,genres,tconst,final_genre,directors,nconst,primaryName
0,1970,Adam at Six A.M.,American,Robert Scheerer,"Michael Douglas, Lee Purcell, Joe Don Baker, L...",unknown,https://en.wikipedia.org/wiki/Adam_at_Six_A.M.,"The film revolves around Adam Gaines, a semant...","The film revolves around Adam Gaines, a semant...",Adam at Six A.M.,1970.0,Drama,tt0065371,Drama,nm0770474,nm0770474,Robert Scheerer
1,1970,The Adventurers,American,Lewis Gilbert,"Bekim Fehmiu, Candice Bergen, Charles Aznavour...",unknown,https://en.wikipedia.org/wiki/The_Adventurers_...,Set in the fictional Latin American country of...,Set in the fictional Latin American country of...,The Adventurers,1970.0,"Action,Adventure,Drama",tt0065374,"Action,Adventure,Drama",nm0318150,nm0318150,Lewis Gilbert
2,1970,Airport,American,George Seaton,"Burt Lancaster, Dean Martin, Jean Seberg, Jacq...",unknown,https://en.wikipedia.org/wiki/Airport_(1970_film),Chicago is paralyzed by a snowstorm affecting ...,Chicago is paralyzed by a snowstorm affecting ...,Airport,1970.0,"Action,Drama,Thriller",tt0065377,"Action,Drama,Thriller","nm0780833,nm0368871",NaN,NaN
3,1970,Alex in Wonderland,American,Paul Mazursky,"Donald Sutherland, Ellen Burstyn, Federico Fel...",unknown,https://en.wikipedia.org/wiki/Alex_in_Wonderland,Young director Alex Morrison feels compelled t...,Alex Morrison feels compelled to follow his re...,Alex in Wonderland,1970.0,"Comedy,Drama",tt0065380,"Comedy,Drama",nm0005196,nm0005196,Paul Mazursky
4,1970,Angel Unchained,American,Lee Madden,"Larry Bishop, Tyne Daly, Aldo Ray",unknown,https://en.wikipedia.org/wiki/Angel_Unchained,"Following a gang fight, biker Angel, calls it ...","Following a gang fight, biker Angel, calls it ...",Angel Unchained,1970.0,"Action,Drama,Thriller",tt0065401,"Action,Drama,Thriller",nm0534613,nm0534613,Lee Madden


In [41]:
### Fallback to plot-dataset Directors name

df_merged_genre['final_director'] = df_merged_genre['primaryName'].combine_first(df_merged_genre['Director'])
df_merged_genre.head(5)

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot,PlotSummary,primaryTitle,startYear,genres,tconst,final_genre,directors,nconst,primaryName,final_director
0,1970,Adam at Six A.M.,American,Robert Scheerer,"Michael Douglas, Lee Purcell, Joe Don Baker, L...",unknown,https://en.wikipedia.org/wiki/Adam_at_Six_A.M.,"The film revolves around Adam Gaines, a semant...","The film revolves around Adam Gaines, a semant...",Adam at Six A.M.,1970.0,Drama,tt0065371,Drama,nm0770474,nm0770474,Robert Scheerer,Robert Scheerer
1,1970,The Adventurers,American,Lewis Gilbert,"Bekim Fehmiu, Candice Bergen, Charles Aznavour...",unknown,https://en.wikipedia.org/wiki/The_Adventurers_...,Set in the fictional Latin American country of...,Set in the fictional Latin American country of...,The Adventurers,1970.0,"Action,Adventure,Drama",tt0065374,"Action,Adventure,Drama",nm0318150,nm0318150,Lewis Gilbert,Lewis Gilbert
2,1970,Airport,American,George Seaton,"Burt Lancaster, Dean Martin, Jean Seberg, Jacq...",unknown,https://en.wikipedia.org/wiki/Airport_(1970_film),Chicago is paralyzed by a snowstorm affecting ...,Chicago is paralyzed by a snowstorm affecting ...,Airport,1970.0,"Action,Drama,Thriller",tt0065377,"Action,Drama,Thriller","nm0780833,nm0368871",NaN,NaN,George Seaton
3,1970,Alex in Wonderland,American,Paul Mazursky,"Donald Sutherland, Ellen Burstyn, Federico Fel...",unknown,https://en.wikipedia.org/wiki/Alex_in_Wonderland,Young director Alex Morrison feels compelled t...,Alex Morrison feels compelled to follow his re...,Alex in Wonderland,1970.0,"Comedy,Drama",tt0065380,"Comedy,Drama",nm0005196,nm0005196,Paul Mazursky,Paul Mazursky
4,1970,Angel Unchained,American,Lee Madden,"Larry Bishop, Tyne Daly, Aldo Ray",unknown,https://en.wikipedia.org/wiki/Angel_Unchained,"Following a gang fight, biker Angel, calls it ...","Following a gang fight, biker Angel, calls it ...",Angel Unchained,1970.0,"Action,Drama,Thriller",tt0065401,"Action,Drama,Thriller",nm0534613,nm0534613,Lee Madden,Lee Madden


In [42]:
# Checking missing values in the df_merged_genre

(df_merged_genre.isna() | (df_merged_genre == "Unknown")| (df_merged_genre == "unknown")|(df_merged_genre == "None")).sum()

# only 95 unknown directors are remaining ( from the initial 126)

Release Year           0
Title                  1
Origin/Ethnicity       0
Director             126
Cast                 163
Genre                307
Wiki Page              0
Plot                   0
PlotSummary            0
primaryTitle        1611
startYear           1610
genres              1610
tconst              1610
final_genre           55
directors           1610
nconst              2018
primaryName         2018
final_director        95
dtype: int64

In [43]:
# Cleaning up df_merged_genre - Keeping what is needed

df_merged_genre.drop(columns=["Director", "Genre","Wiki Page","primaryTitle","startYear","genres","directors","nconst","primaryName"], inplace=True)

df_merged_genre.head(5)

,Release Year,Title,Origin/Ethnicity,Cast,Plot,PlotSummary,tconst,final_genre,final_director
0,1970,Adam at Six A.M.,American,"Michael Douglas, Lee Purcell, Joe Don Baker, L...","The film revolves around Adam Gaines, a semant...","The film revolves around Adam Gaines, a semant...",tt0065371,Drama,Robert Scheerer
1,1970,The Adventurers,American,"Bekim Fehmiu, Candice Bergen, Charles Aznavour...",Set in the fictional Latin American country of...,Set in the fictional Latin American country of...,tt0065374,"Action,Adventure,Drama",Lewis Gilbert
2,1970,Airport,American,"Burt Lancaster, Dean Martin, Jean Seberg, Jacq...",Chicago is paralyzed by a snowstorm affecting ...,Chicago is paralyzed by a snowstorm affecting ...,tt0065377,"Action,Drama,Thriller",George Seaton
3,1970,Alex in Wonderland,American,"Donald Sutherland, Ellen Burstyn, Federico Fel...",Young director Alex Morrison feels compelled t...,Alex Morrison feels compelled to follow his re...,tt0065380,"Comedy,Drama",Paul Mazursky
4,1970,Angel Unchained,American,"Larry Bishop, Tyne Daly, Aldo Ray","Following a gang fight, biker Angel, calls it ...","Following a gang fight, biker Angel, calls it ...",tt0065401,"Action,Drama,Thriller",Lee Madden


In [44]:
#Keeping only actors category from the df_cast 
df_cast = df_cast[df_cast.category=='actor']

# appending actor names 
df_cast = df_cast.merge(df_director_name[['nconst','primaryName']], left_on =['nconst'],right_on=['nconst'],how='left')

## Keeping only tconst,	ordering, primaryName

df_cast = df_cast[["tconst","ordering", "primaryName"]]

df_cast.head(5)

,tconst,ordering,primaryName
0,tt0000005,1,Charles Kayser
1,tt0000005,2,John Ott
2,tt0000007,1,James J. Corbett
3,tt0000007,2,Peter Courtney
4,tt0000008,1,Fred Ott


In [45]:
### Format multi-valued fields - creating comman seperate values

#Combining  primaryNames ( Actors names)  into comma-separated values for each movie 
df_agg = df_cast.groupby('tconst').agg({
    #'ordering': lambda x: ','.join(map(str, x)),
    'primaryName': lambda x: ','.join(map(str,x))
}).reset_index()

# Renaming primaryName to cast_imdb
df_agg.rename(columns={'primaryName': 'cast_imdb'}, inplace=True)
df_agg.head(5)


,tconst,cast_imdb
0,tt0000005,"Charles Kayser,John Ott"
1,tt0000007,"James J. Corbett,Peter Courtney"
2,tt0000008,Fred Ott
3,tt0000009,"William Courtenay,Chauncey Depew"
4,tt0000011,Grunato


In [46]:
### adding cast_imdb to df_merged_genre

df_merged_genre =  df_merged_genre.merge(df_agg, on= 'tconst', how='left')

df_merged_genre.head(5)

,Release Year,Title,Origin/Ethnicity,Cast,Plot,PlotSummary,tconst,final_genre,final_director,cast_imdb
0,1970,Adam at Six A.M.,American,"Michael Douglas, Lee Purcell, Joe Don Baker, L...","The film revolves around Adam Gaines, a semant...","The film revolves around Adam Gaines, a semant...",tt0065371,Drama,Robert Scheerer,"Michael Douglas,Joe Don Baker,Charles Aidman,D..."
1,1970,The Adventurers,American,"Bekim Fehmiu, Candice Bergen, Charles Aznavour...",Set in the fictional Latin American country of...,Set in the fictional Latin American country of...,tt0065374,"Action,Adventure,Drama",Lewis Gilbert,"Charles Aznavour,Alan Badel,Thommy Berggren,Er..."
2,1970,Airport,American,"Burt Lancaster, Dean Martin, Jean Seberg, Jacq...",Chicago is paralyzed by a snowstorm affecting ...,Chicago is paralyzed by a snowstorm affecting ...,tt0065377,"Action,Drama,Thriller",George Seaton,"Burt Lancaster,Dean Martin,George Kennedy,Van ..."
3,1970,Alex in Wonderland,American,"Donald Sutherland, Ellen Burstyn, Federico Fel...",Young director Alex Morrison feels compelled t...,Alex Morrison feels compelled to follow his re...,tt0065380,"Comedy,Drama",Paul Mazursky,"Donald Sutherland,Andre Philippe,Michael Lerne..."
4,1970,Angel Unchained,American,"Larry Bishop, Tyne Daly, Aldo Ray","Following a gang fight, biker Angel, calls it ...","Following a gang fight, biker Angel, calls it ...",tt0065401,"Action,Drama,Thriller",Lee Madden,"Don Stroud,Luke Askew,Larry Bishop,T. Max Grah..."


In [47]:
### Fallback to plot-dataset cast (Actors) name

#picking up cast name from imdb_cast if available, else using Cast
df_merged_genre['final_cast'] = df_merged_genre['cast_imdb'].combine_first(df_merged_genre['Cast'])
df_merged_genre.head(5)


,Release Year,Title,Origin/Ethnicity,Cast,Plot,PlotSummary,tconst,final_genre,final_director,cast_imdb,final_cast
0,1970,Adam at Six A.M.,American,"Michael Douglas, Lee Purcell, Joe Don Baker, L...","The film revolves around Adam Gaines, a semant...","The film revolves around Adam Gaines, a semant...",tt0065371,Drama,Robert Scheerer,"Michael Douglas,Joe Don Baker,Charles Aidman,D...","Michael Douglas,Joe Don Baker,Charles Aidman,D..."
1,1970,The Adventurers,American,"Bekim Fehmiu, Candice Bergen, Charles Aznavour...",Set in the fictional Latin American country of...,Set in the fictional Latin American country of...,tt0065374,"Action,Adventure,Drama",Lewis Gilbert,"Charles Aznavour,Alan Badel,Thommy Berggren,Er...","Charles Aznavour,Alan Badel,Thommy Berggren,Er..."
2,1970,Airport,American,"Burt Lancaster, Dean Martin, Jean Seberg, Jacq...",Chicago is paralyzed by a snowstorm affecting ...,Chicago is paralyzed by a snowstorm affecting ...,tt0065377,"Action,Drama,Thriller",George Seaton,"Burt Lancaster,Dean Martin,George Kennedy,Van ...","Burt Lancaster,Dean Martin,George Kennedy,Van ..."
3,1970,Alex in Wonderland,American,"Donald Sutherland, Ellen Burstyn, Federico Fel...",Young director Alex Morrison feels compelled t...,Alex Morrison feels compelled to follow his re...,tt0065380,"Comedy,Drama",Paul Mazursky,"Donald Sutherland,Andre Philippe,Michael Lerne...","Donald Sutherland,Andre Philippe,Michael Lerne..."
4,1970,Angel Unchained,American,"Larry Bishop, Tyne Daly, Aldo Ray","Following a gang fight, biker Angel, calls it ...","Following a gang fight, biker Angel, calls it ...",tt0065401,"Action,Drama,Thriller",Lee Madden,"Don Stroud,Luke Askew,Larry Bishop,T. Max Grah...","Don Stroud,Luke Askew,Larry Bishop,T. Max Grah..."


In [48]:
### Checking missing values in the df_merged_genre - cast (Actors)
(df_merged_genre.isna() | (df_merged_genre == "Unknown")| (df_merged_genre == "unknown")|(df_merged_genre == "None")).sum()
# 93 unknow cast ( initial 163 )

Release Year           0
Title                  1
Origin/Ethnicity       0
Cast                 163
Plot                   0
PlotSummary            0
tconst              1610
final_genre           55
final_director        95
cast_imdb           1638
final_cast            93
dtype: int64

In [49]:
### Dropping unncessary columns
df_merged_genre.drop(columns=["Cast","cast_imdb","Origin/Ethnicity","tconst"], inplace=True)
df_merged_genre.head()

,Release Year,Title,Plot,PlotSummary,final_genre,final_director,final_cast
0,1970,Adam at Six A.M.,"The film revolves around Adam Gaines, a semant...","The film revolves around Adam Gaines, a semant...",Drama,Robert Scheerer,"Michael Douglas,Joe Don Baker,Charles Aidman,D..."
1,1970,The Adventurers,Set in the fictional Latin American country of...,Set in the fictional Latin American country of...,"Action,Adventure,Drama",Lewis Gilbert,"Charles Aznavour,Alan Badel,Thommy Berggren,Er..."
2,1970,Airport,Chicago is paralyzed by a snowstorm affecting ...,Chicago is paralyzed by a snowstorm affecting ...,"Action,Drama,Thriller",George Seaton,"Burt Lancaster,Dean Martin,George Kennedy,Van ..."
3,1970,Alex in Wonderland,Young director Alex Morrison feels compelled t...,Alex Morrison feels compelled to follow his re...,"Comedy,Drama",Paul Mazursky,"Donald Sutherland,Andre Philippe,Michael Lerne..."
4,1970,Angel Unchained,"Following a gang fight, biker Angel, calls it ...","Following a gang fight, biker Angel, calls it ...","Action,Drama,Thriller",Lee Madden,"Don Stroud,Luke Askew,Larry Bishop,T. Max Grah..."


In [50]:
### Replacing Unknown from final_director, unknown from final_genre, and None from final_cast with NaN

# Replace specific values with NaN
df_merged_genre.replace({
    'final_director': {'Unknown': np.nan},
    'final_genre': {'unknown': np.nan},
    'final_cast': {None: np.nan}
}, inplace=True)

df_merged_genre.head()

,Release Year,Title,Plot,PlotSummary,final_genre,final_director,final_cast
0,1970,Adam at Six A.M.,"The film revolves around Adam Gaines, a semant...","The film revolves around Adam Gaines, a semant...",Drama,Robert Scheerer,"Michael Douglas,Joe Don Baker,Charles Aidman,D..."
1,1970,The Adventurers,Set in the fictional Latin American country of...,Set in the fictional Latin American country of...,"Action,Adventure,Drama",Lewis Gilbert,"Charles Aznavour,Alan Badel,Thommy Berggren,Er..."
2,1970,Airport,Chicago is paralyzed by a snowstorm affecting ...,Chicago is paralyzed by a snowstorm affecting ...,"Action,Drama,Thriller",George Seaton,"Burt Lancaster,Dean Martin,George Kennedy,Van ..."
3,1970,Alex in Wonderland,Young director Alex Morrison feels compelled t...,Alex Morrison feels compelled to follow his re...,"Comedy,Drama",Paul Mazursky,"Donald Sutherland,Andre Philippe,Michael Lerne..."
4,1970,Angel Unchained,"Following a gang fight, biker Angel, calls it ...","Following a gang fight, biker Angel, calls it ...","Action,Drama,Thriller",Lee Madden,"Don Stroud,Luke Askew,Larry Bishop,T. Max Grah..."


In [51]:
### Removing remaining records with missing core metadata

df_american_movies = df_merged_genre.dropna()
df_american_movies.shape 

# (8691, 7) --> 173 movies are deleted 

(8691, 7)

In [59]:
### Validating no missing values, All titles betweem 1970 to 2017, No duplicate (Title, ReleaseYear) pairs remaining 

# ealiest and latest movies in the dataset
print('Earliest Movie:', min(df_american_movies['Release Year']), 'Latest Movie:', max(df_american_movies['Release Year']))

# return 0 if no No duplicate (Title, ReleaseYear) pairs remaining 
print( 'Number of duplicatesTitle, ReleaseYear)', df_Imdb_genre.duplicated(subset=['startYear', 'primaryTitle']).sum())

# return 0 if no missing values
df_american_movies.isna().sum() 


Earliest Movie: 1970 Latest Movie: 2017
Number of duplicatesTitle, ReleaseYear) 0


Release Year      0
Title             0
Plot              0
PlotSummary       0
final_genre       0
final_director    0
final_cast        0
dtype: int64

In [60]:
### saving the cleaned data in a csv file to use for remaining steps

## Saving df_american_movies into a csv file for next steps

df_american_movies.to_csv("df_american_movies_post1969.csv", index=False)

